# Cette partie permet de lancer et utiliser Ollama avec Kaggle

In [ ]:
import os
os.chdir('/kaggle/working/')

#if not os.path.exists('/kaggle/tmp'):
#    os.mkdir('/kaggle/tmp')
#os.chdir('/kaggle/tmp/')

print(os.getcwd())

import subprocess
import os

def run(commands):
    for command in commands:
        with subprocess.Popen(command, shell = True, stdout = subprocess.PIPE, stderr = subprocess.STDOUT, bufsize = 1) as sp:
            for line in sp.stdout:
                line = line.decode("utf-8", errors = "replace")
                if "undefined reference" in line:
                    raise RuntimeError("Failed Processing.")
                print(line, flush = True, end = "")
        pass
    pass
pass



In [ ]:
!pip install ollama

In [ ]:
!ollama

In [ ]:
commands = [
        "ollama pull mistral:7b",
        "ollama pull llama2:7b",
        "ollama pull deepseek-r1",
        ]
run(commands)

In [ ]:
commands = [
        "curl -fsSL https://ollama.com/install.sh | sh",
]
run(commands)

import os
os.system("/usr/local/bin/ollama serve &")
os.system("echo 'ollama test'")

In [ ]:
!pip install dataset

# HellaSwag

In [ ]:
import pandas as pd
import ollama
import time
import warnings
import re
import random
from tqdm import tqdm 
from datasets import load_dataset # Hugging Face datasets library

warnings.filterwarnings("ignore")

# --- Configuration ---
MODEL_NAMES = ["llama2:7b", "mistral:7b", "deepseek-r1"] 


NUM_SHOTS = 0           
NUM_QUESTIONS_TO_PROCESS = 500 
RANDOM_STATE = 40
DATASET_NAME = "Rowan/hellaswag"
DATASET_SPLIT = "validation" 

def extract_predicted_index(text: str) -> int | None:
    """
    Extracts the last predicted index (0, 1, 2, or 3) from the model's response string.

    Args:
        text: The raw response string from the model.

    Returns:
        The extracted integer (0, 1, 2, or 3) if found, otherwise None.
    """
    if not isinstance(text, str):
        return None

    # Find all occurrences of standalone digits 0, 1, 2, or 3.
    # The \b ensures that we match '3' in "option 3" or "3."
    # but not the '3' in "30" or "item3".
    matches = re.findall(r'\b([0-3])\b', text)

    if matches:
        # Get the last match from the list of found matches
        last_match_str = matches[-1]
        try:
            return int(last_match_str)
        except ValueError:
            return None
    
    # Return None if no valid index (0-3) is found
    return None


# --- Main Loop for Each Model ---
for model_name in MODEL_NAMES:
    print(f"\n--- Starting 0-Shot Evaluation for Model: {model_name} ---")
    output_filename_base = f"{model_name.replace(':','-')}_hellaswag" 

    # --- Load Data ---
    print("Loading HellaSwag dataset...")

    # Load validation split for testing
    eval_dataset = load_dataset(DATASET_NAME, split=DATASET_SPLIT)

    # --- Prepare Test Data ---
    # Determine the number of questions to process
    if NUM_QUESTIONS_TO_PROCESS == -1 or NUM_QUESTIONS_TO_PROCESS > len(eval_dataset):
        num_to_process = len(eval_dataset)
    else:
        num_to_process = NUM_QUESTIONS_TO_PROCESS

    # Select the slice of the dataset to process
    eval_data_to_process = eval_dataset.select(range(num_to_process)) # Process first N
    print(f"Loaded {len(eval_dataset)} validation examples. Processing {num_to_process}.")


    # --- System Prompt Setup (0-Shot) ---
    # Instruct the model clearly on the task and desired output format, WITHOUT examples.
    system_prompt = (
        "You are an AI assistant evaluating sentence completions based on commonsense reasoning. "
        "Given a context and four possible endings labeled 0, 1, 2, and 3, "
        "determine which ending is the most logical and natural continuation of the context. "
        "Your response MUST be only the single digit corresponding to the index (0, 1, 2, or 3) of the best ending. "
        "Do not provide any explanation or introductory text. Just the index number."
    )


    # --- Main Processing Loop with tqdm ---
    responses = []
    print(f"Starting processing with model: {model_name}")

    for example in tqdm(eval_data_to_process, desc=f"Processing {model_name}"):
        context = example['ctx']
        endings = example['endings']
        formatted_endings = "\n".join([f"Ending {i}: {endings[i]}" for i in range(len(endings))])

        # User prompt containing the current test question
        user_prompt = (
            f"Context: {context}\n{formatted_endings}\nCorrect Ending Index:" 
        )

        try:
            response = ollama.chat(
                model=model_name,
                messages=[
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user', 'content': user_prompt}
                ],
                options={'temperature': 0.0} # Low temp for deterministic choice prediction
            )
            answer = response["message"]["content"].strip()

        except Exception as e:
            print(f"\nError processing context: '{context[:50]}...'. Model: {model_name}. Error: {e}")
            answer = f"Error: {e}"

        responses.append(answer)

    # --- Post-processing ---
    print(f"\nProcessing complete for {model_name}.")

    # Prepare data for DataFrame
    contexts = [ex['ctx'] for ex in eval_data_to_process]
    all_endings = [ex['endings'] for ex in eval_data_to_process]
    true_labels = [int(ex['label']) for ex in eval_data_to_process] 

    # Extract predicted indices
    predicted_indices = [extract_predicted_index(resp) for resp in responses]

    result_df = pd.DataFrame({
        'context': contexts,
        'endings': all_endings,
        'response_raw': responses,
        'predicted_index': predicted_indices,
        'true_label': true_labels
    })

    # Correct if prediction is not None and matches the true label
    result_df['correct'] = result_df.apply(
        lambda row: row['predicted_index'] is not None and \
                    row['predicted_index'] == row['true_label'],
        axis=1
    )


    # Save to CSV
    output_filename = f"{output_filename_base}.csv"
    result_df.to_csv(output_filename, index=False)
    print(f"✅ Results for {model_name} saved to {output_filename}")

print("\n--- All Model Evaluations Complete ---")

--- Starting 0-Shot Evaluation for Model: llama2:7b ---
Loading HellaSwag dataset...
Loaded 10042 validation examples. Processing 500.
Starting processing with model: llama2:7b
Processing llama2:7b: 100%|██████████| 500/500 [00:40<00:00, 14.15it/s]

Processing complete for llama2:7b.

✅ Results for llama2:7b saved to llama2-7b_hellaswag.csv

--- Starting 0-Shot Evaluation for Model: mistral:7b ---
Loading HellaSwag dataset...
Loaded 10042 validation examples. Processing 500.
Starting processing with model: mistral:7b
Processing mistral:7b: 100%|██████████| 500/500 [00:50<00:00, 14.44it/s]

Processing complete for mistral:7b.

✅ Results for mistral:7b saved to mistral-7b_hellaswag.csv

--- Starting 0-Shot Evaluation for Model: deepseek-r1 ---
Loading HellaSwag dataset...
Loaded 10042 validation examples. Processing 500.
Starting processing with model: deepseek-r1
Processing deepseek-r1: 100%|██████████| 500/500 [1:17:20<00:00,  9.28s/it]

Processing complete for deepseek-r1.

✅ Results 